# TSLA ARIMA-GARCH — frozen historical benchmark and prospective forecasts

Mean: constant drift on daily log returns (random walk with drift in log price); variance: GARCH(1,1) with Student-t shocks, selected by `arima_garch_hyperparameter_optimization.py` on training and validation only (validation QLIKE).

Intervals use **standardized-residual calibration**: validation log errors are divided by the model's forecast volatility, and their empirical quantiles are rescaled by the volatility forecast at each origin. Interval width therefore changes over time. Do not tune using test results.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, *cwd.parents)
                     if (p / "pyproject.toml").is_file()
                     and (p / "shared_code").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from within timeseriesproj")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared_code.visualization.exporting import (
    save_figure,
    save_table,
)

FIGURES_DIR = PROJECT_ROOT / "stock_pred" / "figures" / "arima_garch"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

from shared_code.models.arima_garch import ARIMAGARCHModel
from shared_code.evaluation.diagnostics import training_diagnostic_table, ljung_box_test
from shared_code.evaluation.metrics import interval_summary, compare_point_forecasts
from shared_code.visualization.plotting import (
    plot_price_return_exploration, plot_training_acf_pacf,
    plot_backtest_comparison, plot_backtest_errors,
    plot_validation_error_diagnostics, plot_test_intervals_and_errors,
    plot_predictive_cdf, plot_predictive_distribution,
)
from stock_pred.src.full_stock_pred_pipeline import run_forecasting_pipeline

MODEL_CLASS = ARIMAGARCHModel
# From results/hyperparameter_optimization/best_arima_garch.json (validation QLIKE).
MODEL_KWARGS = {"ar_order": 0, "garch_p": 1, "garch_q": 1, "dist": "t"}
from shared_code.backtest.splits import fixed_date_split
from shared_code.evaluation.predictive_distribution import EmpiricalPredictiveDistribution
from stock_pred.src.split_config import (
    TRAIN_END_DATE, VALIDATION_END_DATE, TEST_END_DATE,
    HORIZON, WINDOW_SIZE, INTERVAL_LEVEL,
)
WINDOW_SIZE = 750  # GARCH-specific rolling window (see best_arima_garch.json)
CSV_PATH = PROJECT_ROOT / "stock_pred" / "data" / "tsla_historical_data.csv"
SPLITS_DIR = PROJECT_ROOT / "stock_pred" / "data" / "fixed_splits"
RESULTS_DIR = PROJECT_ROOT / "stock_pred" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Dates come from stock_pred/src/split_config.py (shared by all four notebooks).
print("Train cutoff:", TRAIN_END_DATE.date())
print("Validation cutoff:", VALIDATION_END_DATE.date())
print("Frozen test cutoff:", TEST_END_DATE.date())


In [ ]:
from stock_pred.src.data.import_data import update_market_data

updated_df, updated_at = update_market_data(
    ticker="TSLA",
    start_date="2021-11-11",
    interval="1d",
)

print("Market data updated at:", updated_at)
print(updated_df[["Date", "Close"]].tail())

## 1. Load full observed history and the fixed benchmark splits

First run `python -m stock_pred.src.prepare_fixed_splits` from the project root. The three CSVs are **frozen benchmark inputs**, not separate independent training histories. Backtesting still needs earlier observed prices when generating later validation/test forecasts.


In [ ]:
raw = pd.read_csv(CSV_PATH)

# Extract the trading date, ignoring the timezone offset.
raw["Date"] = pd.to_datetime(
    raw["Date"].astype(str).str[:10],
    format="%Y-%m-%d",
    errors="raise"
)

prices = (
    raw.sort_values("Date")
       .drop_duplicates("Date", keep="last")
       .set_index("Date")["Close"]
       .astype(float)
)

assert isinstance(prices.index, pd.DatetimeIndex)
assert prices.index.is_monotonic_increasing
assert prices.index.is_unique

print(
    f"{len(prices)} observations, "
    f"latest: {prices.index[-1].date()}"
)

display(prices.tail())

def load_fixed_csv(name):
    path = SPLITS_DIR / f"tsla_{name}.csv"
    frame = pd.read_csv(path, parse_dates=["Date"])
    return frame.set_index("Date")["Close"].astype(float).rename("Close")

fixed_train = load_fixed_csv("train")
fixed_validation = load_fixed_csv("validation")
fixed_test = load_fixed_csv("test")
historical_prices = pd.concat([fixed_train, fixed_validation, fixed_test])

# Detect mismatched config/CSV or accidentally refreshed historical splits.
expected = fixed_date_split(prices, TRAIN_END_DATE, VALIDATION_END_DATE, TEST_END_DATE)
for actual, desired in zip((fixed_train, fixed_validation, fixed_test), expected):
    pd.testing.assert_series_equal(actual, desired.rename("Close"), check_freq=False)

train_end = len(fixed_train)
validation_end = train_end + len(fixed_validation)
assert historical_prices.index.max() <= TEST_END_DATE
print(f"Frozen benchmark: {len(fixed_train)} train / {len(fixed_validation)} validation / {len(fixed_test)} test")
print("Full latest observation:", prices.index[-1].date())


## Exploratory statistics and return volatility
These plots describe the full observed history; formal pre-selection diagnostics below use only the initial training partition.

In [ ]:
log_returns = np.log(prices).diff().dropna()
summary = pd.DataFrame({"close": prices.describe(), "log_return": log_returns.describe()})
display(summary)
fig, axes = plot_price_return_exploration(prices)
plt.show()

## 2. Exploratory plot

In [ ]:
ax = prices.plot(figsize=(10, 4), title="TSLA closing prices")
ax.set_ylabel("Closing price ($)")
plt.show()

## Training-only stationarity and dependence diagnostics

Only the frozen initial training partition is used here. Do not use test-period evidence to select ARIMA orders or MLP hyperparameters.


In [ ]:
train_prices = fixed_train
train_log_returns = np.log(train_prices).diff().dropna()
diagnostic_table = training_diagnostic_table(train_prices)
display(diagnostic_table.round(5))
fig, axes = plot_training_acf_pacf(train_prices)
plt.show()

### Model configuration

Values copied from `stock_pred/results/hyperparameter_optimization/best_arima_garch.json`. GARCH uses a 750-session rolling window (the other models use 252): with 252 sessions the GARCH estimates often hit the alpha = 0, beta = 1 boundary, so volatility would not respond to recent shocks.

In [ ]:
print('Model:', MODEL_CLASS.__name__)
print('Parameters:', MODEL_KWARGS)

In [ ]:
# Training-only AIC/BIC/log-likelihood and parameters for the configured ARIMA-GARCH model
garch_train_model = MODEL_CLASS(**MODEL_KWARGS)
garch_train_model.fit(fixed_train.iloc[-WINDOW_SIZE:].to_numpy(dtype=float))

garch_information_criteria = pd.DataFrame([garch_train_model.information_criteria()],
                                          index=pd.Index(["ARIMA-GARCH(1,1)"], name="Model"))
display(garch_information_criteria.round(3))
display(garch_train_model.results.params.rename("estimate (returns in %)").to_frame().round(4))
params = garch_train_model.results.params
print(f"Volatility persistence alpha + beta = {params['alpha[1]'] + params['beta[1]']:.4f}")

## 3. Run the frozen historical backtest

The backtest uses the concatenated fixed CSVs as one ordered history. Validation and test **forecast targets** are fixed, while each rolling-origin fit may use observations preceding its target date.


In [ ]:
results = run_forecasting_pipeline(
    prices=historical_prices,
    model_class=MODEL_CLASS,
    model_kwargs=MODEL_KWARGS,
    horizon=HORIZON,
    window_size=WINDOW_SIZE,
    train_end=train_end,
    validation_end=validation_end,
    interval_level=INTERVAL_LEVEL,
)
print("Model:", results.model_name)
print("Train:", results.train.index[0].date(), "to", results.train.index[-1].date())
print("Validation:", results.validation["target_date"].iloc[0], "to", results.validation["target_date"].iloc[-1])
print("Test:", results.test["target_date"].iloc[0], "to", results.test["target_date"].iloc[-1])
assert results.validation["target_date"].tolist() == list(fixed_validation.index)
assert results.test["target_date"].tolist() == list(fixed_test.index)


## Rolling-origin backtests: validation and test
Detailed origin/target rows, and separate actual-vs-forecast and forecast-error panels.

In [ ]:
columns = ["origin_date", "target_date", "train_start", "train_size", "y_true", "y_pred", "error"]
print("VALIDATION BACKTEST")
display(results.validation[columns].head(8))
print("TEST BACKTEST")
display(results.test[columns].head(8))
for bt in (results.validation, results.test):
    assert (bt["horizon"] == 1).all()
    assert (bt["train_size"] == min(WINDOW_SIZE, len(results.train))).all()
    assert (bt["origin_date"] < bt["target_date"]).all()
assert set(results.validation["target_date"]).isdisjoint(set(results.test["target_date"]))
fig, axes = plot_backtest_comparison(results.validation, results.test)
plt.show()
fig, axes = plot_backtest_errors(results.validation, results.test)
plt.show()

## 4. Validation diagnostics and calibration errors

`results.validation_diagnostics` is computed on the **standardized** validation errors z = log(actual / forecast) / sigma. A large ARCH p-value means the GARCH volatility forecast has absorbed the volatility clustering; std_error should be close to 1.

In [ ]:
display(results.validation_diagnostics)
display(results.validation.head())
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(results.validation["error"], bins=30)
ax.set_title("Validation one-step-ahead forecast errors (actual − predicted)")
plt.show()

### Validation forecast-error diagnostics and dependence plots

In [ ]:
errors = results.validation.set_index("target_date")["error"].astype(float)
lb_lags = [lag for lag in (5, 10) if lag < len(errors)]
display(ljung_box_test(errors, lags=lb_lags))
fig, axes = plot_validation_error_diagnostics(results.validation)
plt.show()

## 5. Held-out test evaluation

In [ ]:
display(results.evaluation)
check = results.test.copy()
coverage_summary = interval_summary(check, level=INTERVAL_LEVEL)
point_comparison = compare_point_forecasts(check)
display(coverage_summary.round(4))
display(point_comparison.round(4))
display(check[["origin_date", "target_date", "y_true", "y_pred", "predictive_mean", "interval_lower", "interval_upper", "covered"]].head(10))
fig, axes = plot_test_intervals_and_errors(check, level=INTERVAL_LEVEL)
plt.show()

## 6. Latest next-session forecast (separate from frozen historical evaluation)

Fit the selected model specification on the latest completed 750 trading-day closes. Reuse the **frozen standardized validation errors**, rescaled by today's GARCH volatility forecast, to construct the empirical predictive distribution. The frozen test CSVs and their metrics are not updated.


In [ ]:
# verification
# Verify that the data is current before live forecasting.
from stock_pred.src.data.import_data import (
    latest_completed_nyse_session
)

latest_completed = latest_completed_nyse_session()
latest_observed = pd.Timestamp(prices.index[-1]).normalize()

assert latest_observed == latest_completed, (
    f"Stale dataset: latest observed={latest_observed.date()}, "
    f"latest completed NYSE session={latest_completed.date()}"
)

print(
    "Live forecast data verified through:",
    latest_observed.date()
)


# Refit the fixed model specification on the latest available prices.
live_model = MODEL_CLASS(**MODEL_KWARGS)
live_model.fit(prices.iloc[-WINDOW_SIZE:].to_numpy(dtype=float))
live_point = float(np.asarray(live_model.predict(HORIZON), dtype=float)[-1])
live_scale = float(np.asarray(live_model.predict_scale(HORIZON), dtype=float)[-1])
live_distribution = results.predictive_distribution(live_point, live_scale)
print(f"GARCH one-step volatility forecast: {live_scale:.4%} of price")
live_lower, live_upper = live_distribution.interval(INTERVAL_LEVEL)

# Determine whether this can be recorded as a genuinely prospective forecast.
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas_market_calendars as mcal

nyse = mcal.get_calendar("NYSE")
now = pd.Timestamp(datetime.now(ZoneInfo("America/New_York")))
origin = pd.Timestamp(prices.index[-1]).normalize()

# NYSE schedules include holidays and early closing days.
schedule = nyse.schedule(
    start_date=min(origin, now.tz_localize(None).normalize()) - pd.Timedelta(days=20),
    end_date=max(origin, now.tz_localize(None).normalize()) + pd.Timedelta(days=20),
)
completed = schedule.loc[schedule["market_close"] <= now.tz_convert("UTC")]
if completed.empty:
    raise RuntimeError("No completed NYSE session found in calendar window")
latest_completed_session = pd.Timestamp(completed.index[-1]).normalize()

future_sessions = schedule.index[schedule.index > origin]
if len(future_sessions) == 0:
    raise RuntimeError("Could not identify the next NYSE session")
target_date = pd.Timestamp(future_sessions[0]).normalize()
target_close = schedule.loc[target_date, "market_close"]

is_prospective = bool(
    origin >= TEST_END_DATE
    and target_date > TEST_END_DATE
    and origin == latest_completed_session
    and now.tz_convert("UTC") < target_close
)

if not is_prospective:
    print("NOT ELIGIBLE FOR LIVE HISTORY: forecast is stale or target is already closed.")
    print(f"Latest CSV close: {origin.date()} | Latest completed NYSE session: {latest_completed_session.date()}")
    print(f"Target: {target_date.date()} | Current NY time: {now}")
else:
    print(f"Eligible prospective forecast: {origin.date()} -> {target_date.date()}")

live_forecast = pd.DataFrame([{
    "origin_date": origin,
    "target_date": target_date,
    "model": MODEL_CLASS.__name__,
    "y_true": np.nan,
    "y_pred": live_point,
    "predictive_mean": live_distribution.mean(),
    "interval_lower": live_lower,
    "interval_upper": live_upper,
    "interval_level": INTERVAL_LEVEL,
    "last_close": float(prices.iloc[-1]),
}])
display(live_forecast)
ax = plot_predictive_cdf(live_distribution)
plt.show()

### Empirical predictive sample distribution

In [ ]:
fig, axes = plot_predictive_distribution(live_distribution, level=INTERVAL_LEVEL)
plt.show()


## 7. Export standardized results for `main.ipynb`

The frozen historical test results are always exported. A prospective forecast is appended **only if** the origin is the latest completed NYSE session and the target session has not closed. An existing forecast is never overwritten. Do not backfill old forecasts as if they were issued live.

In [ ]:
EXPORT_COLUMNS = [
    "origin_date",
    "target_date",
    "model",
    "y_true",
    "y_pred",
    "predictive_mean",
    "interval_lower",
    "interval_upper",
]

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------
# 1. Preserve the frozen historical benchmark
# ---------------------------------------------

test_path = RESULTS_DIR / "arima_garch_test.csv"

if not test_path.exists():
    results.test[EXPORT_COLUMNS].to_csv(
        test_path, index=False
    )
    print("Created frozen historical test results.")
else:
    print("Historical test results already frozen; not overwritten.")


# ---------------------------------------------
# 2. Load prospective forecast history
# ---------------------------------------------

history_path = RESULTS_DIR / "arima_garch_live_history.csv"

if history_path.exists():
    history = pd.read_csv(history_path)
else:
    history = pd.DataFrame(columns=EXPORT_COLUMNS)

# Normalize dates to ensure correct matching
for col in ["origin_date", "target_date"]:
    history[col] = pd.to_datetime(history[col]).dt.normalize()

actual_prices = prices.copy()
actual_prices.index = pd.to_datetime(actual_prices.index).normalize()

# ---------------------------------------------
# 3. Reconcile actual observed closing prices
# ---------------------------------------------

observed_actuals = history["target_date"].map(actual_prices)

history["y_true"] = pd.to_numeric(
    history["y_true"], errors="coerce"
).combine_first(observed_actuals)

# ---------------------------------------------
# 4. Append new eligible prospective forecast
# ---------------------------------------------

if is_prospective:

    new_forecast = live_forecast[EXPORT_COLUMNS].copy()

    key_cols = ["model", "origin_date", "target_date"]

    duplicate = (
        (history["model"] == MODEL_CLASS.__name__)
        & (history["origin_date"] == origin)
        & (history["target_date"] == target_date)
    ).any()

    if duplicate:
        print("Forecast already recorded; preserving original.")
    else:
        history = pd.concat(
            [history, new_forecast],
            ignore_index=True
        )
        print("New prospective forecast appended.")

else:
    print("No new eligible prospective forecast.")

# ---------------------------------------------
# 5. Save updated prospective history
# ---------------------------------------------

history = history.sort_values(
    ["origin_date", "target_date"]
).reset_index(drop=True)

history.to_csv(history_path, index=False)

print("Saved:", history_path)
display(history.tail())